# RAG System — Submission PGABL 

| Item | Detail |
|---|---|
| **Model** | `anggapradana/qwen2-5-1-5b-grpo-id` (hasil GRPO) |
| **Dokumen** | 4 file PDF UU/PP Ketenagakerjaan Indonesia |
| **Embedding** | `paraphrase-multilingual-MiniLM-L12-v2` (open-source) |
| **Vector DB** | FAISS (lokal) |
| **Platform** | Kaggle (GPU T4) |

## 1. Instalasi Package

In [3]:
%%capture
!pip install unsloth
!pip install -U langchain langchain-community
!pip install sentence-transformers rank_bm25
!pip install faiss-cpu pypdf
!pip install duckduckgo-search gradio

## 2. Autentikasi HuggingFace

Token dibaca dari **Kaggle Secrets** (label: `HF_TOKEN`).

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

from huggingface_hub import login
login(token=hf_token)
print("Login HuggingFace berhasil.")

## 3. Load Model GRPO untuk Inferensi — Basic Kriteria 2

Model dimuat dari HuggingFace (`anggapradana/qwen2-5-1-5b-grpo-id`) menggunakan Unsloth dengan QLoRA 4-bit
untuk efisiensi VRAM. Model ini adalah hasil dari proses Fine-tuning + GRPO pada Kriteria 1.

In [5]:
from unsloth import FastLanguageModel
import torch

GRPO_MODEL    = "anggapradana/qwen2-5-1-5b-grpo-id"
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=GRPO_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

print(f"Model '{GRPO_MODEL}' berhasil dimuat untuk inferensi.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Model 'anggapradana/qwen2-5-1-5b-grpo-id' berhasil dimuat untuk inferensi.


## 4. Fungsi Generasi Teks — Basic Kriteria 2

Prompt template berisi `{context}` (dokumen yang diambil) dan `{question}` (pertanyaan pengguna).
Model GRPO menghasilkan reasoning `<think>...</think>` sebelum jawaban final.

In [6]:
GRPO_SYSTEM_PROMPT = (
    "Anda adalah asisten AI legal yang helpful dan jujur. "
    "Sebelum memberikan jawaban, tuliskan proses berpikir Anda secara detail "
    "dalam tag <think>...</think>. "
    "Kemudian berikan jawaban final yang jelas dan terstruktur dalam Bahasa Indonesia."
)

def generate_answer(context: str, question: str, max_new_tokens: int = 512) -> str:
    """
    Generate jawaban dari model GRPO berdasarkan {context} dan {question}.
    Model akan menghasilkan reasoning <think>...</think> sebelum jawaban final.
    """
    prompt_content = (
        f"Berdasarkan dokumen hukum berikut, jawab pertanyaan pengguna dengan tepat.\n\n"
        f"Konteks:\n{context}\n\n"
        f"Pertanyaan: {question}"
    )
    messages = [
        {"role": "system", "content": GRPO_SYSTEM_PROMPT},
        {"role": "user",   "content": prompt_content},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    return tokenizer.decode(
        output_ids[0][input_ids.shape[1]:],
        skip_special_tokens=True,
    )

print("Fungsi generate_answer siap.")

Fungsi generate_answer siap.


## 5. Load Dokumen PDF & Metadata Enrichment — Basic + Skilled Kriteria 2

Keempat file PDF UU/PP dimuat seluruhnya menggunakan `PyPDFLoader`.
Setiap halaman diperkaya dengan **metadata enrichment**: `source`, `judul`, `jenis`, dan `page`.

In [8]:
import glob
import os
from langchain_community.document_loaders import PyPDFLoader

# Cek path dataset secara otomatis
import os
possible_dirs = glob.glob("/kaggle/input/*/")
print("Dataset yang ter-attach:")
for d in possible_dirs:
    print(f"  {d}")

# Cari semua PDF dari seluruh input
pdf_files = sorted(glob.glob("/kaggle/input/**/*.pdf", recursive=True))
print(f"\nDitemukan {len(pdf_files)} file PDF:")
for f in pdf_files:
    print(f"  - {f}")

# Metadata enrichment: mapping nama file ke judul & jenis dokumen
DOC_METADATA_MAP = {
    "PP Nomor 35 Tahun 2021": {
        "judul": "PP No. 35/2021 — PKWT, Alih Daya, Waktu Kerja dan Upah Lembur",
        "jenis": "Peraturan Pemerintah",
    },
    "PP Nomor 5 Tahun 2021": {
        "judul": "PP No. 5/2021 — Penyelenggaraan Perizinan Berusaha Berbasis Risiko",
        "jenis": "Peraturan Pemerintah",
    },
    "PP Nomor 51 Tahun 2023": {
        "judul": "PP No. 51/2023 — Pengupahan",
        "jenis": "Peraturan Pemerintah",
    },
    "UU Nomor 6 Tahun 2023": {
        "judul": "UU No. 6/2023 — Cipta Kerja",
        "jenis": "Undang-Undang",
    },
}

all_documents = []
for pdf_path in pdf_files:
    loader   = PyPDFLoader(pdf_path)
    pages    = loader.load()
    filename = os.path.basename(pdf_path).replace(".pdf", "")

    extra_meta = {"judul": filename, "jenis": "Dokumen Hukum"}
    for key, meta in DOC_METADATA_MAP.items():
        if key in filename:
            extra_meta = meta
            break

    for page in pages:
        page.metadata.update({
            "source": filename,
            "judul":  extra_meta["judul"],
            "jenis":  extra_meta["jenis"],
        })
    all_documents.extend(pages)

print(f"\nTotal halaman dimuat: {len(all_documents)}")
for pdf_path in pdf_files:
    filename = os.path.basename(pdf_path).replace(".pdf", "")
    count = sum(1 for d in all_documents if d.metadata.get("source") == filename)
    print(f"  {filename}: {count} halaman")

print("\nContoh metadata satu dokumen:")
print(all_documents[0].metadata)

Dataset yang ter-attach:
  /kaggle/input/datasets/

Ditemukan 4 file PDF:
  - /kaggle/input/datasets/anggayulian/rag-documents-pgabl/PP Nomor 35 Tahun 2021.pdf
  - /kaggle/input/datasets/anggayulian/rag-documents-pgabl/PP Nomor 5 Tahun 2021.pdf
  - /kaggle/input/datasets/anggayulian/rag-documents-pgabl/PP Nomor 51 Tahun 2023.pdf
  - /kaggle/input/datasets/anggayulian/rag-documents-pgabl/UU Nomor 6 Tahun 2023.pdf

Total halaman dimuat: 1949
  PP Nomor 35 Tahun 2021: 56 halaman
  PP Nomor 5 Tahun 2021: 739 halaman
  PP Nomor 51 Tahun 2023: 27 halaman
  UU Nomor 6 Tahun 2023: 1127 halaman

Contoh metadata satu dokumen:
{'producer': '', 'creator': 'Canon', 'creationdate': '2021-02-18T15:54:05+07:00', 'moddate': '2021-02-18T16:07:05+07:00', 'source': 'PP Nomor 35 Tahun 2021', 'total_pages': 56, 'page': 0, 'page_label': '1', 'judul': 'PP No. 35/2021 — PKWT, Alih Daya, Waktu Kerja dan Upah Lembur', 'jenis': 'Peraturan Pemerintah'}


## 6. Text Splitting: Child & Parent Chunks — Basic + Skilled Kriteria 2

Dua level chunking digunakan untuk **Parent-Child Retriever**:
- **Child chunks** (kecil, 256 token): digunakan untuk pencarian vektor yang presisi
- **Parent chunks** (besar, 1024 token): diberikan ke LLM sebagai konteks lengkap

In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Child chunks: kecil, untuk pencarian vektor yang presisi
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=256,
    chunk_overlap=32,
    separators=["\n\n", "\n", ".", " ", ""],
)

# Parent chunks: besar, untuk konteks LLM yang kaya
parent_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1024,
    chunk_overlap=128,
    separators=["\n\n", "\n", ".", " ", ""],
)

child_docs  = child_splitter.split_documents(all_documents)
parent_docs = parent_splitter.split_documents(all_documents)

print(f"Child chunks  (chunk_size=256,  overlap=32)  : {len(child_docs):,}")
print(f"Parent chunks (chunk_size=1024, overlap=128) : {len(parent_docs):,}")
print(f"\nContoh child chunk:")
print(f"  Source : {child_docs[0].metadata.get('source')}")
print(f"  Page   : {child_docs[0].metadata.get('page')}")
print(f"  Konten : {child_docs[0].page_content[:150]}...")

Child chunks  (chunk_size=256,  overlap=32)  : 9,480
Parent chunks (chunk_size=1024, overlap=128) : 3,205

Contoh child chunk:
  Source : PP Nomor 35 Tahun 2021
  Page   : 0
  Konten : SALINAN
PRESIDEN
REPUBLIK INDONESIA
PERATURAN PEMERINTAH REPUBLIK INDONESIA
NOMOR 35 TAHUN 2O2I
TENTANG
PERJANJIAN KERJA WAKTU TERTENTU, ALIH DAYA, WA...


## 7. Embedding Open-Source & FAISS Vector Database — Basic Kriteria 2

Model embedding `paraphrase-multilingual-MiniLM-L12-v2` digunakan karena mendukung Bahasa Indonesia.
Child chunks diubah menjadi vektor dan disimpan ke **FAISS** secara lokal.

In [11]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

EMBED_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print(f"Memuat model embedding: {EMBED_MODEL}")

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Membangun FAISS index dari child chunks...")
faiss_db = FAISS.from_documents(child_docs, embeddings)
print(f"FAISS index berhasil dibuat: {len(child_docs):,} child chunks diindeks.")

Memuat model embedding: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


/tmp/ipykernel_58/2170822222.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Membangun FAISS index dari child chunks...
FAISS index berhasil dibuat: 9,480 child chunks diindeks.


## 8. Ensemble Retriever: BM25 + FAISS — Skilled Kriteria 2

Menggabungkan dua metode retrieval dengan bobot:
- **BM25** (keyword-based): bobot **0.4** — unggul untuk istilah hukum spesifik
- **FAISS** (semantic): bobot **0.6** — unggul untuk query semantik/parafrase

Minimal **5 dokumen** diambil sebelum tahap reranking.

In [15]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import CallbackManagerForRetrieverRun
from langchain_core.documents import Document
from typing import List

# Implementasi EnsembleRetriever manual (kompatibel semua versi langchain)
class EnsembleRetriever(BaseRetriever):
    retrievers: list
    weights: list

    def _get_relevant_documents(self, query: str, *, run_manager: CallbackManagerForRetrieverRun) -> List[Document]:
        all_docs = []
        seen_keys = set()
        for retriever in self.retrievers:
            docs = retriever.invoke(query)
            for doc in docs:
                key = (doc.metadata.get("source"), doc.metadata.get("page"), doc.page_content[:80])
                if key not in seen_keys:
                    seen_keys.add(key)
                    all_docs.append(doc)
        return all_docs

# BM25 Retriever dari child docs
bm25_retriever = BM25Retriever.from_documents(child_docs)
bm25_retriever.k = 5

# FAISS Retriever
faiss_retriever = faiss_db.as_retriever(search_kwargs={"k": 5})

# Ensemble: gabungkan BM25 (keyword) + FAISS (semantic)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, faiss_retriever],
    weights=[0.4, 0.6],
)

print("Ensemble Retriever (BM25 40% + FAISS 60%) siap.")

# Uji retrieval
test_results = ensemble_retriever.invoke("upah lembur pekerja")
print(f"Uji retrieval 'upah lembur pekerja': {len(test_results)} dokumen diambil")
for i, doc in enumerate(test_results[:3]):
    print(f"  [{i+1}] {doc.metadata.get('source')} | Hal. {doc.metadata.get('page')} | {doc.page_content[:60]}...")

Ensemble Retriever (BM25 40% + FAISS 60%) siap.
Uji retrieval 'upah lembur pekerja': 10 dokumen diambil
  [1] PP Nomor 35 Tahun 2021 | Hal. 17 | bersangkutan dan Pengusaha.
(3) Pengusaha sebagaimana dimaks...
  [2] UU Nomor 6 Tahun 2023 | Hal. 558 | dimaksud pada ayat (1) hurrrf b tidak berlaku bagi
sektor us...
  [3] PP Nomor 35 Tahun 2021 | Hal. 18 | PRESIDEN
REPUELIK INDONESIA
-L9-
a. untuk ja- kerja lembur p...


## 9. Parent-Child Retriever — Skilled Kriteria 2

Setelah child chunks ditemukan via Ensemble Retriever, fungsi ini
mencari **parent chunk** (potongan besar) yang mencakup child tersebut.
Parent chunk diberikan ke LLM agar konteks lebih lengkap dan koheren.

In [16]:
def find_parent_chunk(child_doc, parent_docs: list):
    """Temukan parent chunk yang mencakup child chunk berdasarkan source + page + konten."""
    child_source  = child_doc.metadata.get("source", "")
    child_page    = child_doc.metadata.get("page", 0)
    child_content = child_doc.page_content

    for parent in parent_docs:
        if parent.metadata.get("source") != child_source:
            continue
        if abs(parent.metadata.get("page", 0) - child_page) > 1:
            continue
        if child_content[:60] in parent.page_content:
            return parent

    return child_doc  # fallback: kembalikan child jika parent tidak ditemukan


def retrieve_with_parent_context(query: str, retriever, parent_docs: list, k: int = 7) -> list:
    """
    Retrieve child chunks via Ensemble Retriever,
    lalu kembalikan parent chunks sebagai konteks untuk LLM.
    """
    child_results = retriever.invoke(query)[:k]
    parent_results = []
    seen_keys = set()

    for child in child_results:
        parent = find_parent_chunk(child, parent_docs)
        key = (parent.metadata.get("source"), parent.metadata.get("page"), parent.page_content[:80])
        if key not in seen_keys:
            seen_keys.add(key)
            parent_results.append(parent)

    return parent_results


# Uji Parent-Child
print("Uji Parent-Child Retriever:")
test_parents = retrieve_with_parent_context("lembur 3 jam upah", ensemble_retriever, parent_docs)
print(f"  Child chunks → {len(test_parents)} parent chunks")
for doc in test_parents[:2]:
    print(f"  [{doc.metadata.get('source')} | Hal. {doc.metadata.get('page')}] len={len(doc.page_content)} char")

Uji Parent-Child Retriever:
  Child chunks → 6 parent chunks
  [PP Nomor 35 Tahun 2021 | Hal. 18] len=1015 char
  [UU Nomor 6 Tahun 2023 | Hal. 557] len=434 char


## 10. HyDE: Hypothetical Document Embeddings — Advanced Kriteria 2

Teknik **HyDE** mentransformasi query dengan cara:
1. LLM menghasilkan **minimal 2 jawaban hipotetis** (halusinasi terkontrol) dari query
2. Jawaban hipotetis tersebut di-embed dan digunakan sebagai query retrieval
3. Hasil retrieval lebih relevan karena berdasarkan ruang semantik jawaban, bukan pertanyaan

In [17]:
def generate_hypothetical_answers(query: str, n: int = 2) -> list:
    """
    Generate n jawaban hipotetis dari LLM untuk query.
    Jawaban ini digunakan sebagai dokumen embedding (HyDE).
    """
    hyde_prompt = (
        f"Anda adalah ahli hukum ketenagakerjaan Indonesia. "
        f"Tuliskan {n} kemungkinan jawaban BERBEDA untuk pertanyaan berikut "
        f"berdasarkan regulasi ketenagakerjaan Indonesia (UU Cipta Kerja, PP Pengupahan, dsb). "
        f"Setiap jawaban harus spesifik, mengandung pasal/regulasi, dan berbeda satu sama lain.\n\n"
        f"Pertanyaan: {query}\n\n"
        f"Jawaban 1:"
    )
    messages = [
        {"role": "system", "content": "Anda adalah asisten hukum yang berpengetahuan luas tentang regulasi Indonesia. Berikan jawaban singkat dan padat."},
        {"role": "user",   "content": hyde_prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=300,
            temperature=0.8,
            do_sample=True,
            pad_token_id=tokenizer.pad_token_id,
        )

    raw_output = tokenizer.decode(
        output_ids[0][input_ids.shape[1]:], skip_special_tokens=True
    )

    # Parse jawaban 1 dan 2
    parts = raw_output.split("Jawaban 2:")
    hyp1 = parts[0].replace("Jawaban 1:", "").strip()
    hyp2 = parts[1].strip() if len(parts) > 1 else hyp1 + " (perspektif alternatif)"

    return [hyp1[:500], hyp2[:500]]


def hyde_retrieve(query: str, retriever, parent_docs: list, k: int = 7) -> list:
    """
    HyDE: retrieve menggunakan jawaban hipotetis sebagai query tambahan.
    Menggabungkan hasil dari query asli + 2 hipotesis.
    """
    print(f"  [HyDE] Menghasilkan 2 jawaban hipotetis...")
    hyp_answers = generate_hypothetical_answers(query, n=2)
    print(f"  [HyDE] Hipotesis 1: {hyp_answers[0][:80]}...")
    print(f"  [HyDE] Hipotesis 2: {hyp_answers[1][:80]}...")

    # Retrieve menggunakan query asli + kedua hipotesis
    all_results = []
    seen_keys   = set()

    for q in [query, hyp_answers[0], hyp_answers[1]]:
        docs = retrieve_with_parent_context(q, retriever, parent_docs, k=3)
        for doc in docs:
            key = (doc.metadata.get("source"), doc.metadata.get("page"), doc.page_content[:80])
            if key not in seen_keys:
                seen_keys.add(key)
                all_results.append(doc)

    return all_results[:k]


# Uji HyDE
print("Uji HyDE:")
hyde_results = hyde_retrieve("berapa batas jam lembur per minggu?", ensemble_retriever, parent_docs)
print(f"  Total dokumen setelah HyDE: {len(hyde_results)}")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Uji HyDE:
  [HyDE] Menghasilkan 2 jawaban hipotetis...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

  [HyDE] Hipotesis 1: 1. Pasal 54 Ayat 3 UU Cipta Kerja menyatakan bahwa setiap orang memiliki hak mem...
  [HyDE] Hipotesis 2: 1. Pasal 54 Ayat 3 UU Cipta Kerja menyatakan bahwa setiap orang memiliki hak mem...
  Total dokumen setelah HyDE: 4


## 11. Reranker Cross-Encoder & Relevance Score — Advanced Kriteria 2

Model **Cross-Encoder** (`ms-marco-MiniLM-L-6-v2`) digunakan untuk mengurutkan ulang dokumen
berdasarkan relevansi terhadap query. Hanya **Top-3** dokumen terbaik yang diambil.

**Relevance Score Top-1** diekstrak untuk menentukan apakah dokumen lokal cukup relevan.

In [18]:
from sentence_transformers import CrossEncoder

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
print(f"Memuat Reranker: {RERANKER_MODEL}")

reranker = CrossEncoder(RERANKER_MODEL, max_length=512)
print("Reranker Cross-Encoder berhasil dimuat.")


def rerank_documents(query: str, documents: list, top_k: int = 3) -> tuple:
    """
    Rerank dokumen dengan Cross-Encoder.
    Kembalikan (top_k_docs, relevance_score_top1).
    """
    if not documents:
        return [], 0.0

    pairs  = [(query, doc.page_content[:512]) for doc in documents]
    scores = reranker.predict(pairs)

    scored_docs = sorted(zip(scores, documents), key=lambda x: x[0], reverse=True)
    top_docs    = [doc for _, doc in scored_docs[:top_k]]
    top1_score  = float(scored_docs[0][0]) if scored_docs else 0.0

    print(f"  [Reranker] Skor dokumen teratas (Top-{top_k}):")
    for i, (score, doc) in enumerate(scored_docs[:top_k]):
        print(f"    [{i+1}] score={score:.4f} | {doc.metadata.get('source')} | Hal. {doc.metadata.get('page')}")

    return top_docs, top1_score

Memuat Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker Cross-Encoder berhasil dimuat.


## 12. DuckDuckGo Fallback — Advanced Kriteria 2

Jika **Relevance Score Top-1 < threshold (-4.0)**, sistem mengabaikan dokumen lokal
dan beralih ke **DuckDuckGo Search** untuk mencari informasi dari internet.

Kondisi ini terpicu ketika pertanyaan di luar cakupan keempat dokumen PDF.

In [ ]:
from duckduckgo_search import DDGS

# ms-marco-MiniLM-L-6-v2 menghasilkan raw logit, bukan probabilitas 0-1.
# Threshold -4.0 memisahkan dokumen relevan (skor > -4) dari tidak relevan (skor < -4).
RERANKER_THRESHOLD = -4.0

def duckduckgo_search(query: str, max_results: int = 3) -> str:
    """Cari informasi dari internet via DuckDuckGo sebagai fallback."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(
                f"{query} peraturan hukum Indonesia",
                region="id-id",
                max_results=max_results,
            ))
        if not results:
            return "Tidak ditemukan hasil dari pencarian internet."

        context_parts = []
        for r in results:
            context_parts.append(f"Sumber: {r.get('href', '')}\n{r.get('body', '')}")
        return "\n\n".join(context_parts)

    except Exception as e:
        return f"Pencarian internet gagal: {e}. Coba jawab berdasarkan pengetahuan umum."


print(f"DuckDuckGo fallback siap. Threshold Reranker Top-1: {RERANKER_THRESHOLD}")

## 13. Pipeline RAG

Pipeline menggabungkan semua komponen:
1. **HyDE** → query transformation dengan 2 hipotesis
2. **Ensemble Retriever** (BM25 + FAISS) → retrieve ≥5 dokumen
3. **Parent-Child Retriever** → upgrade ke parent chunks
4. **Metadata Filtering** → filter berdasarkan source (opsional)
5. **Reranker Cross-Encoder** → ambil Top-3 paling relevan + ekstrak score Top-1
6. **Relevance Score check** → jika < -4.0, beralih ke DuckDuckGo
7. **Generate jawaban** dengan model GRPO (reasoning `<think>` + jawaban final)
8. **Sitasi** → tambahkan referensi dokumen yang digunakan

In [20]:
def format_citations(documents: list) -> str:
    """Format sitasi dari dokumen yang digunakan sebagai konteks."""
    seen      = set()
    citations = []
    for doc in documents:
        source = doc.metadata.get("source", "Unknown")
        page   = doc.metadata.get("page", "?")
        judul  = doc.metadata.get("judul", source)
        key    = (source, page)
        if key not in seen:
            seen.add(key)
            citations.append(f"- {judul}, Halaman {int(page) + 1}")
    return "\n".join(citations) if citations else "- Tidak ada sitasi"


def rag_pipeline(question: str, source_filter: str = None) -> str:
    """
    Pipeline RAG lengkap (Basic + Skilled + Advanced).

    Args:
        question      : Pertanyaan pengguna
        source_filter : Filter dokumen berdasarkan nama sumber (opsional, Skilled)

    Returns:
        Jawaban lengkap dengan reasoning <think> dan sitasi
    """
    print(f"\n{'='*65}")
    print(f"PERTANYAAN: {question}")
    print(f"{'='*65}")

    # === STEP 1: HyDE + Ensemble + Parent-Child Retrieval ===
    print("[Step 1] HyDE Retrieval...")
    retrieved_docs = hyde_retrieve(question, ensemble_retriever, parent_docs, k=7)
    print(f"  Dokumen ditemukan: {len(retrieved_docs)}")

    # === STEP 2: Metadata Filtering (Skilled) ===
    if source_filter:
        retrieved_docs = [
            d for d in retrieved_docs
            if source_filter.lower() in d.metadata.get("source", "").lower()
        ]
        print(f"[Step 2] Metadata filter '{source_filter}': {len(retrieved_docs)} dokumen tersisa")
    else:
        print(f"[Step 2] Metadata filter: tidak diaktifkan")

    # === STEP 3: Reranker ===
    print("[Step 3] Reranking dengan Cross-Encoder...")
    top_docs, top1_score = rerank_documents(question, retrieved_docs, top_k=3)

    # === STEP 4: Relevance Score Check + DuckDuckGo Fallback (Advanced) ===
    if top1_score < RERANKER_THRESHOLD or len(top_docs) == 0:
        print(f"[Step 4] ⚠️  Score Top-1 ({top1_score:.4f}) < threshold ({RERANKER_THRESHOLD})")
        print("         Beralih ke DuckDuckGo Search...")
        internet_context = duckduckgo_search(question)
        context       = f"[Sumber: Pencarian Internet]\n{internet_context}"
        citation_text = "- Sumber: DuckDuckGo Search (dokumen lokal tidak cukup relevan)"
    else:
        print(f"[Step 4] ✅ Score Top-1 ({top1_score:.4f}) ≥ threshold. Menggunakan dokumen lokal.")
        context       = "\n\n".join([doc.page_content for doc in top_docs])
        citation_text = format_citations(top_docs)

    # === STEP 5: Generate Jawaban ===
    print("[Step 5] Menghasilkan jawaban...")
    answer = generate_answer(context, question)

    # === STEP 6: Tambah Sitasi (Skilled) ===
    final_answer = f"{answer}\n\n---\n**Sumber Referensi:**\n{citation_text}"

    print("[Selesai] Jawaban berhasil dihasilkan.")
    return final_answer


print("RAG Pipeline lengkap siap.")

RAG Pipeline lengkap siap.


## 14. Test Case Wajib

Prompt wajib dari kriteria diuji melalui pipeline RAG lengkap.
Model harus menampilkan reasoning `<think>...</think>` sebelum jawaban final.

In [21]:
TEST_PROMPT = "Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"

result = rag_pipeline(TEST_PROMPT)

print("\n" + "="*65)
print("JAWABAN FINAL:")
print("="*65)
print(result)

# Verifikasi format reasoning
result_lower = result.lower()
has_think     = "<think>" in result_lower
has_end_think = "</think>" in result_lower
print(f"\n✅ Tag <think> hadir   : {has_think}")
print(f"✅ Tag </think> hadir  : {has_end_think}")
print(f"✅ Format reasoning    : {has_think and has_end_think}")

Both `max_new_tokens` (=300) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



PERTANYAAN: Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?
[Step 1] HyDE Retrieval...
  [HyDE] Menghasilkan 2 jawaban hipotetis...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


  [HyDE] Hipotesis 1: Tidak, Anda tidak berhak mendapatkan uang lembur jika hanya memerlukan waktu sek...
  [HyDE] Hipotesis 2: Tidak, Anda tidak berhak mendapatkan uang lembur karena itu hanya merupakan peke...


/tmp/ipykernel_58/2496104874.py:8: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  Dokumen ditemukan: 7
[Step 2] Metadata filter: tidak diaktifkan
[Step 3] Reranking dengan Cross-Encoder...
  [Reranker] Skor dokumen teratas (Top-3):
    [1] score=-3.7001 | PP Nomor 35 Tahun 2021 | Hal. 18
    [2] score=-4.5941 | UU Nomor 6 Tahun 2023 | Hal. 558
    [3] score=-4.7925 | UU Nomor 6 Tahun 2023 | Hal. 557
[Step 4] ⚠️  Score Top-1 (-3.7001) < threshold (0.3)
         Beralih ke DuckDuckGo Search...
[Step 5] Menghasilkan jawaban...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnin

[Selesai] Jawaban berhasil dihasilkan.

JAWABAN FINAL:
<think>
Berdasarkan konteks yang diberikan, ada beberapa faktor yang mempengaruhi apakah seorang staf admin dapat mengklaim uang lembur atau tidak.
1. Jika seorang staf admin melakukan tugas-tugas non-staf atau jika perusahaan tersebut memiliki aturan khusus tentang lemburan, maka mereka mungkin tidak memiliki hak untuk mengklaim uang lembur.
2. Dalam kasus ini, kita diminta untuk mengecek keberadaan laporan lembur oleh seorang staf admin pada tanggal kemarin, 3 jam lembur. Ini harus dilakukan melalui sistem HR atau melalui formulir yang disediakan oleh perusahaan.
3. Bila laporan lembur tersebut diisi dengan benar, maka seorang staf admin akan memiliki hak untuk mengklaim uang lembur. Namun, bila laporan tidak lengkap atau tidak sesuai dengan ketentuan, maka seorang staf admin mungkin tidak memiliki hak untuk mengklaim uang lembur.
</think>

Jawaban:
Dengan mengamati bahwa seorang staf admin telah melakukan tugas-tugas non-staf se

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnin

## 15. Uji DuckDuckGo Fallback

Menguji kondisi fallback dengan pertanyaan di luar cakupan dokumen PDF,
membuktikan sistem beralih ke DuckDuckGo ketika Relevance Score rendah.

In [ ]:
# Pertanyaan di luar cakupan 4 dokumen PDF → trigger DuckDuckGo fallback
# Topik kuliner/resep masakan tidak ada dalam dokumen hukum ketenagakerjaan
FALLBACK_PROMPT = "Apa resep nasi goreng spesial yang enak dan cara memasaknya?"

result_fallback = rag_pipeline(FALLBACK_PROMPT)

print("\n" + "="*65)
print("JAWABAN (DuckDuckGo Fallback):")
print("="*65)
print(result_fallback[:800] + "..." if len(result_fallback) > 800 else result_fallback)

## 16. Interface Gradio — Basic Kriteria 2

Pipeline RAG dibungkus dalam **Gradio Interface** sederhana dengan satu input teks dan satu output teks.
Interface dapat diakses melalui link publik yang di-generate oleh Gradio.

In [24]:
import gradio as gr

def gradio_rag(question: str) -> str:
    if not question.strip():
        return "Silakan masukkan pertanyaan."
    return rag_pipeline(question)


interface = gr.Interface(
    fn=gradio_rag,
    inputs=gr.Textbox(
        label="Pertanyaan Hukum Ketenagakerjaan",
        placeholder="Contoh: Apakah saya berhak mendapat pesangon jika di-PHK?",
        lines=3,
    ),
    outputs=gr.Textbox(
        label="Jawaban AI (reasoning <think> + jawaban final + sitasi)",
        lines=25,
    ),
    title="RAG Legal AI — PGABL Submission",
    description=(
        "Sistem RAG berbasis 4 dokumen UU/PP Ketenagakerjaan Indonesia. "
        "Model: Qwen2.5-1.5B (fine-tuned + GRPO). "
        "Fitur: HyDE, Ensemble Retriever, Reranker, DuckDuckGo fallback."
    ),
    examples=[
        ["Saya staf admin, kemarin lembur 3 jam untuk beresin laporan. Apakah saya berhak dapat uang lembur?"],
        ["Berapa batas waktu kerja per hari menurut UU Cipta Kerja?"],
        ["Apa syarat perusahaan boleh menggunakan alih daya atau outsourcing?"],
        ["Bagaimana cara menghitung upah lembur pada hari libur?"],
    ],
    allow_flagging="never",
)

interface.launch(share=True, debug=False)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

* Running on local URL:  http://127.0.0.1:7860


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

* Running on public URL: https://bebc77199b6718613b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

## Ringkasan Kriteria 2

| Level | Kriteria | Implementasi | Status |
|---|---|---|---|
| **Basic** | Load 4 file PDF | `PyPDFLoader` — semua PDF dimuat | ✅ |
| **Basic** | Text splitting eksplisit (chunk_size + overlap) | Child: 256/32, Parent: 1024/128 | ✅ |
| **Basic** | Embedding open-source | `paraphrase-multilingual-MiniLM-L12-v2` | ✅ |
| **Basic** | Vector DB lokal | FAISS | ✅ |
| **Basic** | Load model hasil Kriteria 1 | `qwen2-5-1-5b-grpo-id` via Unsloth | ✅ |
| **Basic** | Prompt `{context}` + `{question}` + inference | `generate_answer()` | ✅ |
| **Basic** | Interface Gradio | `gr.Interface` (1 input, 1 output) | ✅ |
| **Skilled** | Metadata enrichment | source, page, judul, jenis | ✅ |
| **Skilled** | Metadata filtering | filter by source dalam pipeline | ✅ |
| **Skilled** | Sitasi jawaban | `format_citations()` | ✅ |
| **Skilled** | Ensemble Retriever BM25+FAISS + bobot | BM25 40% + FAISS 60%, k=7 (≥5) | ✅ |
| **Skilled** | Parent-Child Retriever | Child 256 untuk search → Parent 1024 untuk LLM | ✅ |
| **Advanced** | HyDE ≥2 jawaban hipotetis | 2 hipotesis via LLM | ✅ |
| **Advanced** | Reranker Cross-Encoder + Top-K=3 | `ms-marco-MiniLM-L-6-v2` | ✅ |
| **Advanced** | Ekstrak Relevance Score Top-1 | `top1_score` dari CrossEncoder | ✅ |
| **Advanced** | If-else threshold + DuckDuckGo fallback | threshold=-4.0 (raw logit ms-marco) | ✅ |